# S7: test-split evaluation (student epoch3 or B0)

Set `STAGE` to `"student"` or `"b0"` (run the other in a second session). 40 test tasks x 3 trials, split over the two T4s
(20 tasks each). Test mode is locked unless `results/threshold.md` exists in the code bundle.

**Before running:** Accelerator = GPU T4 x2, Internet = On, secrets `HF_TOKEN` and `API_DEEPSEEK`; inputs:
`agentic-distill-src` and the output of kernel `agentic-distill-s5-train`.
DeepSeek peak pricing (Mon-Fri 06:30-09:30 and 11:30-15:30 IST) is refused by the runner: start before Monday 06:30 IST.


In [ ]:
STAGE = "student"   # "student" or "b0"
import sys, subprocess, os, glob, re, hashlib
print(sys.version)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
ADAPTER = None
if STAGE == "student":
    hits = glob.glob("/kaggle/input/**/lora/epoch3/adapter_config.json", recursive=True)
    assert hits, "attach the output of agentic-distill-s5-train"
    ADAPTER = os.path.dirname(hits[0]); print("adapter:", ADAPTER)
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
for label in ("HF_TOKEN", "API_DEEPSEEK"):
    os.environ[label] = s.get_secret(label)          # values never printed
print("secrets loaded:", [k for k in ("HF_TOKEN", "API_DEEPSEEK") if os.environ.get(k)])


In [ ]:
%%bash
set -e
rm -rf /kaggle/working/repo && mkdir -p /kaggle/working/repo
Z=$(find /kaggle/input -name agentic_distill_src.zip | head -1)
if [ -n "$Z" ]; then (cd /kaggle/working/repo && unzip -q "$Z"); \
else D=$(dirname "$(find /kaggle/input -name pyproject.toml | head -1)"); cp -r "$D"/. /kaggle/working/repo/; fi
cd /kaggle/working/repo
ls results/threshold.md
mkdir -p third_party && git clone -q https://github.com/sierra-research/tau2-bench third_party/tau2-bench
git -C third_party/tau2-bench checkout -q c06b07dd7c4d4ec61ba6ff9bcdf0b15dd7ef36d4
pip uninstall -y -q torchao
pip install -q -e third_party/tau2-bench audioop-lts pyyaml
pip install -q -U transformers peft accelerate
python -c "import tau2, transformers, peft, torch; print('ok; transformers', transformers.__version__, 'gpus', torch.cuda.device_count())"


In [ ]:
# The checkpoint must be the one frozen in results/threshold.md.
if STAGE == "student":
    frozen = re.search(r"sha256 `([0-9a-f]{64})`", open("/kaggle/working/repo/results/threshold.md").read()).group(1)
    h = hashlib.sha256(open(os.path.join(ADAPTER, "adapter_model.safetensors"), "rb").read()).hexdigest()
    print("adapter sha256:", h); assert h == frozen, "adapter differs from the one frozen in threshold.md"


In [ ]:
import json, threading
split = json.load(open("/kaggle/working/repo/data/splits/retail_split.json"))
TEST = split["test_ids"]; assert len(TEST) == 40
LABEL = "s2-epoch3" if STAGE == "student" else "b0"
def launch(gpu, tasks):
    env = {**os.environ, "PYTHONPATH": "src", "CUDA_VISIBLE_DEVICES": str(gpu),
           "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
    cmd = ["python", "-m", "agentic_distill.student_run", "--tasks", *tasks, "--split", "test", "--trials", "3",
           "--slice", "eval_student", "--label", LABEL, "--out-root", "/kaggle/working/out"]
    if ADAPTER: cmd += ["--adapter", ADAPTER]
    p = subprocess.Popen(cmd, cwd="/kaggle/working/repo", env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    def pump():
        with open(f"/kaggle/working/{LABEL}_gpu{gpu}.log", "w") as f:
            for line in p.stdout:
                f.write(line); f.flush()
                if line.startswith(("task=", "slice", "STOP", "Traceback")) or "Error" in line[:40]:
                    print(f"[gpu{gpu}] {line}", end="", flush=True)
    t = threading.Thread(target=pump); t.start()
    return p, t
procs = [launch(0, TEST[0::2]), launch(1, TEST[1::2])]
for p, t in procs: p.wait(); t.join()
print("exit codes:", [p.returncode for p, _ in procs])


In [ ]:
%%bash
cd /kaggle/working/repo
PYTHONPATH=src python -m agentic_distill.metrics "x=/kaggle/working/out/data/raw/eval_student_*gpu*" 2>&1 | grep -v DEBUG | head -3 || true
echo; ls /kaggle/working/out/data/raw/
cat /kaggle/working/out/data/raw/*/episodes.jsonl | wc -l
